#Cálculo da WER

In [ ]:
!pip install jiwer re

ERROR: Could not find a version that satisfies the requirement re (from versions: none)
ERROR: No matching distribution found for re


In [ ]:
import jiwer
import re

In [ ]:
_PUNCT_CHAR = r"(?!(?<=\w)['’-](?=\w))[^\w\s]"

_PUNCT_SUB = re.compile(f"({_PUNCT_CHAR})")
_IS_PUNCT = re.compile(r"^[^\w\s]$")
PUNCT = "[PUNCT]"


def normalize_text(text):
    text = text.lower()
    text = re.sub(_PUNCT_CHAR, "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_separate_punctuation(text):
    text = text.lower()
    text = _PUNCT_SUB.sub(r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
def _mask(tokens):
  return [PUNCT if _IS_PUNCT.match(t) else t for t in tokens]


def _align_punctuation(ref, hyp):
  """Alinha as sequências mascaradas e devolve os pares (i, j) de
  pontuações alinhadas entre si."""
  ref_m, hyp_m = _mask(ref), _mask(hyp)
  n, m = len(ref), len(hyp)
  big = n + m + 1

  def step(i, j):
    if ref_m[i] != hyp_m[j]:
      return big
    if ref_m[i] == PUNCT and ref[i] != hyp[j]:
      return 1
    return 0

  dist = [[0] * (m + 1) for _ in range(n + 1)]
  for i in range(1, n + 1):
    dist[i][0] = i * big
  for j in range(1, m + 1):
    dist[0][j] = j * big
  for i in range(1, n + 1):
    for j in range(1, m + 1):
        dist[i][j] = min(
          dist[i - 1][j - 1] + step(i - 1, j - 1),
          dist[i - 1][j] + big,
          dist[i][j - 1] + big,
        )

  pairs, i, j = [], n, m
  while i > 0 and j > 0:
    if dist[i][j] == dist[i - 1][j - 1] + step(i - 1, j - 1):
      if ref_m[i - 1] == PUNCT and hyp_m[j - 1] == PUNCT:
          pairs.append((i - 1, j - 1))
      i, j = i - 1, j - 1
    elif dist[i][j] == dist[i - 1][j] + big:
      i -= 1
    else:
      j -= 1
  return pairs


def punctuation_counts(reference, hypothesis):
  """Espera textos já passados por normalize_separate_punctuation."""
  ref, hyp = reference.split(), hypothesis.split()
  pairs = _align_punctuation(ref, hyp)

  n_ref = sum(1 for t in ref if _IS_PUNCT.match(t))
  n_hyp = sum(1 for t in hyp if _IS_PUNCT.match(t))
  correct = sum(ref[i] == hyp[j] for i, j in pairs)
  return {
    "C": correct,
    "S": len(pairs) - correct,
    "D": n_ref - len(pairs),
    "I": n_hyp - len(pairs),
  }


def _per(c):
  errors = c["S"] + c["D"] + c["I"]
  total = errors + c["C"]
  return errors / total if total else 0.0


def punctuation_error_rate(reference, hypothesis):
  """
  Cálculo da Punctuation Error Rate (PER), com base em https://arxiv.org/abs/2310.02943

  PER = Ip + Dp + Sp / Ip + Dp + Sp + Cp
  Ip = Pontuações Inseridas
  Dp = Pontuações Deletadas
  Sp = Pontuações Substituídas
  Cp = Pontuações Corretas
  """
  reference = normalize_separate_punctuation(reference)
  hypothesis = normalize_separate_punctuation(hypothesis)
  return _per(punctuation_counts(reference, hypothesis))


def corpus_punctuation_error_rate(references, hypotheses):
  """ PER de um corpus todo"""
  total = {"C": 0, "S": 0, "D": 0, "I": 0}
  for r, h in zip(references, hypotheses):
    for k, v in punctuation_counts(r, h).items():
      total[k] += v
  return _per(total)

In [ ]:
reference_string = "O rato a. roeu. a roupa do rei de Roma."
transcribed_string = "O rato. roeu , a roupa do rei de roma."

ref_clean = normalize_text(reference_string)
hyp_clean = normalize_text(transcribed_string)

print(ref_clean)
print(hyp_clean)
print("WER:", jiwer.wer(ref_clean, hyp_clean))
print("CER:", jiwer.cer(ref_clean, hyp_clean))

ref_punc = normalize_separate_punctuation(reference_string)
hyp_punc = normalize_separate_punctuation(transcribed_string)

print(ref_punc)
print(hyp_punc)
print("WER:", jiwer.wer(ref_punc, hyp_punc))
print("CER:", jiwer.cer(ref_punc, hyp_punc))

print("PER:", punctuation_error_rate(ref_punc, hyp_punc))

o rato a roeu a roupa do rei de roma
o rato roeu a roupa do rei de roma
WER: 0.1
CER: 0.05555555555555555
o rato a . roeu . a roupa do rei de roma .
o rato . roeu , a roupa do rei de roma .
WER: 0.15384615384615385
CER: 0.07142857142857142
PER: 0.3333333333333333
